# 03 · Memory — make TravelMind remember
### short-term vs long-term · the four strategies · a memory hook · branches

In notebook 02 you saw that *within* a Runtime session the agent remembers (same warm microVM). The moment the session ends, that's gone. **AgentCore Memory** is the durable, cross-session, queryable memory — and it doesn't just store transcripts, it **extracts** facts, preferences, and summaries for you.

Two layers:

- **Short-term memory (STM)** — the raw recent turns of a conversation. Fast, literal, "what was just said."
- **Long-term memory (LTM)** — extracted knowledge written into **namespaces** by background **strategies**. Semantic, durable, "what we know about this traveler."

```mermaid
flowchart LR
    E[create_event<br/>raw turns] --> STM[Short-term: get_last_k_turns]
    E --> X[Strategies extract async]
    X --> S1[Semantic facts]
    X --> S2[User preferences]
    X --> S3[Rolling summary]
    S1 & S2 & S3 --> LTM[Long-term: retrieve_memories]
```

> Prereqs: notebook 00. Memory provisioning takes a minute, and **LTM extraction is asynchronous** — facts appear a short while after you write events, not instantly.


## The four strategies (what gets extracted)

A **strategy** is a background extractor you attach to a memory store. Each writes into namespaces you template with `{actorId}` / `{sessionId}`:

- **`semanticMemoryStrategy`** — durable facts ("traveler holds a UK passport").
- **`userPreferenceMemoryStrategy`** — stable preferences ("aisle seat, vegetarian meal").
- **`summaryMemoryStrategy`** — a rolling summary of the conversation.
- **`customMemoryStrategy`** / `episodicMemoryStrategy` — your own extraction prompt / event-style memories.

You pick the ones your use case needs. TravelMind wants **preferences** (seat/meal) and **semantic facts** (loyalty number, home airport).


In [ ]:
import os, time
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"
REGION = "us-east-1"

from bedrock_agentcore.memory import MemoryClient

mc = MemoryClient(region_name=REGION)

# Strategy dicts are keyed by the strategy type. Namespaces use {actorId}/{sessionId} templates.
strategies = [
    {"userPreferenceMemoryStrategy": {
        "name": "TravelerPrefs",
        "namespaces": ["travelmind/{actorId}/preferences"],
    }},
    {"semanticMemoryStrategy": {
        "name": "TravelerFacts",
        "namespaces": ["travelmind/{actorId}/facts"],
    }},
]

memory = mc.create_memory_and_wait(            # blocks until ACTIVE (about a minute)
    name="TravelMindMemory",
    strategies=strategies,
    description="Durable traveler preferences + facts for TravelMind",
    event_expiry_days=90,                      # raw events auto-expire after 90 days
)
MEMORY_ID = memory.get("memoryId") or memory.get("id")
print("Memory ready:", MEMORY_ID)

## 1. Short-term — write turns, read them back

`create_event` records a turn. Messages are **`(text, role)`** tuples (roles: `USER`, `ASSISTANT`, `TOOL`, `OTHER`). An `actor_id` identifies the traveler; a `session_id` groups one conversation.


In [ ]:
ACTOR = "traveler-amelia"
SESSION = "trip-2026-06-tokyo"

mc.create_event(
    memory_id=MEMORY_ID, actor_id=ACTOR, session_id=SESSION,
    messages=[
        ("I always want an aisle seat and a vegetarian meal.", "USER"),
        ("Noted — aisle seat and vegetarian meal on every booking.", "ASSISTANT"),
    ],
)
mc.create_event(
    memory_id=MEMORY_ID, actor_id=ACTOR, session_id=SESSION,
    messages=[
        ("My home airport is LHR and my loyalty number is BA-99812.", "USER"),
        ("Got it. Home airport LHR, loyalty BA-99812.", "ASSISTANT"),
    ],
)

# Short-term read: the most recent raw turns
recent = mc.get_last_k_turns(memory_id=MEMORY_ID, actor_id=ACTOR, session_id=SESSION, k=3)
print("Recent turns (STM):")
for turn in recent:
    print("  ", turn)

## 2. Long-term — let strategies extract, then retrieve

The events above feed the strategies, which **asynchronously** distill preferences/facts into the namespaces. After a short wait, `retrieve_memories` does semantic recall against a namespace.

`retrieve_memories` needs **exactly one** of `namespace` (exact) or `namespace_path` (prefix), plus a `query`.


In [ ]:
# Extraction is async — give it a moment in a live run (poll/retry in production).
time.sleep(20)

prefs = mc.retrieve_memories(
    memory_id=MEMORY_ID,
    namespace=f"travelmind/{ACTOR}/preferences",   # matches the strategy template, resolved for this actor
    query="seat and meal preferences",
    top_k=3,
)
print("Long-term preferences recalled:")
for rec in prefs:
    print("  ", rec)

facts = mc.retrieve_memories(
    memory_id=MEMORY_ID,
    namespace=f"travelmind/{ACTOR}/facts",
    query="home airport and loyalty number",
    top_k=3,
)
print("\nLong-term facts recalled:")
for rec in facts:
    print("  ", rec)

## 3. Wire memory into a Strands agent — a memory **hook**

Manually calling `create_event` / `retrieve_memories` works, but you want it automatic. Strands **hooks** let you run code at lifecycle moments. We register two callbacks:

- **on every message added** → persist it (`create_event`). This builds STM and feeds LTM.
- **on agent init** → recall the traveler's long-term preferences and inject them into the system prompt, so the agent *starts* already knowing them.

A `HookProvider` just implements `register_hooks(registry)` and calls `registry.add_callback(EventType, fn)`.


In [ ]:
from strands import Agent, tool
from strands.hooks import HookProvider, HookRegistry, MessageAddedEvent, AgentInitializedEvent

def _text_of(message) -> str:
    # A Strands message is {"role": ..., "content": [{"text": ...}, ...]}
    return "".join(b.get("text", "") for b in message.get("content", []) if isinstance(b, dict)).strip()

class MemoryHook(HookProvider):
    """Auto-save every turn to AgentCore Memory and preload long-term preferences on startup."""
    def __init__(self, client, memory_id, actor_id, session_id):
        self.client, self.memory_id = client, memory_id
        self.actor_id, self.session_id = actor_id, session_id

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(AgentInitializedEvent, self.on_init)
        registry.add_callback(MessageAddedEvent, self.on_message)

    def on_init(self, event: AgentInitializedEvent) -> None:
        recalled = self.client.retrieve_memories(
            memory_id=self.memory_id,
            namespace=f"travelmind/{self.actor_id}/preferences",
            query="traveler seat and meal preferences",
            top_k=5,
        )
        if recalled:
            notes = "; ".join(str(r) for r in recalled)
            event.agent.system_prompt += f"\n\nKnown traveler preferences: {notes}"

    def on_message(self, event: MessageAddedEvent) -> None:
        role = event.message.get("role", "").upper()
        text = _text_of(event.message)
        if role in ("USER", "ASSISTANT") and text:
            self.client.create_event(
                memory_id=self.memory_id, actor_id=self.actor_id,
                session_id=self.session_id, messages=[(text, role)],
            )

@tool
def get_flight_status(flight_number: str) -> dict:
    """Look up the current status of a flight by its flight number (e.g. BA117)."""
    db = {"BA117": {"status": "Delayed", "delay_min": 75}, "AA100": {"status": "On time"}}
    return db.get(flight_number.upper(), {"status": "Unknown flight"})

agent = Agent(
    model="us.anthropic.claude-haiku-4-5-20251001-v1:0",
    tools=[get_flight_status],
    system_prompt="You are TravelMind. Be concise and use known preferences when relevant.",
    hooks=[MemoryHook(mc, MEMORY_ID, ACTOR, SESSION)],   # <-- memory, fully wired
)

# The agent starts already knowing Amelia's preferences (loaded on init); the turn is saved too.
print(str(agent("Book me a flight to Tokyo — anything you should already know about my seat/meal?")))

That's the whole pattern: in a *new* session next week, a fresh agent with the same hook reloads Amelia's preferences and continues as if it never forgot.

> **Even less code:** `strands_tools` ships `AgentCoreMemoryToolProvider(memory_id, actor_id, session_id, namespace)`, which hands the agent ready-made memory *tools* it can call itself. And on Runtime you can skip wiring entirely with `configure(memory_mode="STM_AND_LTM")` or the CLI's `agentcore add memory`. The hook above is the explicit version so you can see what those shortcuts do.


## 4. Branches — hand data between agents

A **branch** forks a conversation from a point in time. Why it matters for multi-agent work (notebook 05): when an orchestrator spins off a sub-investigation (say, a rebooking thread), you branch so the side-conversation has its own lineage without polluting the main thread — and another agent can read that branch as its handed-over context.

- New branch: `branch={"rootEventId": <event id>, "name": "rebooking"}`.
- Continue a branch: `branch={"name": "rebooking"}`.
- Inspect: `list_branches(...)`, `get_conversation_tree(...)`.


In [ ]:
# Anchor on the latest event, then fork a 'rebooking' branch off it.
root_event_id = recent[0][0].get("eventId") if recent and isinstance(recent[0][0], dict) else None

if root_event_id:
    mc.create_event(
        memory_id=MEMORY_ID, actor_id=ACTOR, session_id=SESSION,
        branch={"rootEventId": root_event_id, "name": "rebooking"},
        messages=[
            ("BA117 is delayed 75 min — find an earlier alternative.", "USER"),
            ("Searching alternatives on the rebooking branch.", "ASSISTANT"),
        ],
    )
    print("Branches:", mc.list_branches(memory_id=MEMORY_ID, actor_id=ACTOR, session_id=SESSION))
else:
    print("No event id available to branch from in this run — write an event first, then re-run.")

## Common errors → fixes

| Symptom | Cause | Fix |
|---|---|---|
| `retrieve_memories` returns `[]` right after writing | LTM extraction is async | Wait/poll; STM (`get_last_k_turns`) is immediate |
| Nothing recalled even after waiting | Read namespace ≠ write namespace | Make `retrieve` namespace match the strategy template (resolved for the actor) |
| `ValueError: Each message must be (text, role)` | Wrong tuple order | It's **(text, role)**, role in {USER, ASSISTANT, TOOL, OTHER} |
| `AccessDenied` creating memory | Missing memory permissions | Ensure your principal/role can call AgentCore Memory APIs |
| Events vanish over time | `event_expiry_days` reached | Raw events are meant to expire; durable knowledge lives in LTM namespaces |

> **Production note:** give Memory its own execution role (least privilege), set `event_expiry_days` to your retention policy, treat LTM extraction as eventually-consistent (poll, don't assume), and key namespaces by a stable user id so the right traveler's memory is recalled.

## Next

**`04_tools_and_identity.ipynb`** — extend the agent's reach: **Gateway** (turn a Lambda/OpenAPI into MCP tools), **Code Interpreter** (sandboxed compute for fare math), **Browser** (live web when there's no API), and **Identity** (who the agent is allowed to act as, and how it authenticates to external APIs).
